In [15]:
# Program to filter bogons from team-cymru
import pandas as pd
import requests
import ipaddress

scrubber = "198949"

# --- Step 1: Download bogons lists ---
def download_bogons(url):
    response = requests.get(url)
    return [
        line.strip() for line in response.text.splitlines()
        if line.strip() and not line.startswith("#")
    ]

ipv4_bogons = download_bogons("https://team-cymru.org/Services/Bogons/fullbogons-ipv4.txt")
ipv6_bogons = download_bogons("https://team-cymru.org/Services/Bogons/fullbogons-ipv6.txt")

ipv4_networks = [ipaddress.ip_network(prefix) for prefix in ipv4_bogons]
ipv6_networks = [ipaddress.ip_network(prefix) for prefix in ipv6_bogons]

# --- Step 2: Read your CSV ---
df = pd.read_csv("../data/merged/as"+scrubber+"_may_duration.csv")

# Make sure your prefix column is named correctly
prefix_column = "Prefix"  # Change if different
df[prefix_column] = df[prefix_column].astype(str)  # Ensure strings

# --- Step 3: Check each prefix against bogons ---
def is_bogon(prefix):
    try:
        net = ipaddress.ip_network(prefix)
        if net.version == 4:
            return any(net.subnet_of(b) or b.subnet_of(net) for b in ipv4_networks)
        elif net.version == 6:
            return any(net.subnet_of(b) or b.subnet_of(net) for b in ipv6_networks)
        else:
            return False
    except ValueError:
        return False

df['is_bogon'] = df[prefix_column].apply(is_bogon)

# --- Step 4: Save results ---
# df.to_csv("checked_prefixes.csv", index=False)
# print("Done! Results saved to checked_prefixes.csv")
print(df[df["is_bogon"] == True])


Empty DataFrame
Columns: [Date, Prefix, StartTime, EndTime, DurationMinutes, DurationSeconds, is_bogon]
Index: []
